In [4]:
from ingest import load_faq_data
documents = load_faq_data()

In [5]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [6]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [7]:
documents = documents_llm

In [8]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
import json
user_prompt = json.dumps(doc)

In [13]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [14]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [15]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [16]:
response.output_parsed.questions

['I just found this course — can I still sign up and take it now?',
 'Am I too late to join the course if I discovered it after it started?',
 'Can I still participate in the course, or is it already closed?',
 'If I join late, can I still get a certificate somehow?',
 'What do I need to do to be eligible for the certificate if I’m joining now?']

In [17]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [18]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

--2026-07-12 01:21:47--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-12 01:21:47 (13.2 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]

--2026-07-12 01:21:47--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting respon

In [19]:
from evaluation_utils import llm_structured

In [20]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I join the course late if I just found it, or is it too late now?', 'Am I still allowed to sign up even though the course has already started?', 'If I join the course now, will I still be eligible for a certificate?', 'What’s the deadline for getting a certificate if I started the course late?', 'Do I have to submit the project before submissions close in order to get certified?']


In [21]:
usage.input_tokens, usage.output_tokens

(207, 97)

In [22]:
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.00015525, 'output_cost': 0.0004365, 'total_cost': 0.00059175}

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I join the course late if I just found it, or is it too late now?',
  'document': '74eb249bbf'},
 {'question': 'Am I still allowed to sign up even though the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, will I still be eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for getting a certificate if I started the course late?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit the project before submissions close in order to get certified?',
  'document': '74eb249bbf'}]

In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [27]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [28]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [30]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost


0.087432

In [31]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [32]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [34]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)